<a href="https://colab.research.google.com/github/Ravi-ranjan1801/CUDA-Lab/blob/main/cuda_lab_08_map_reduce_matrix_mul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile mapreduce_matmul.cu
// mapreduce_matmul.cu
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <time.h>

__global__ void mapReduceMatMul(int *A, int *B, int *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < N && col < N)
    {
        int sum = 0;

        // MAP + REDUCE combined
        for(int k = 0; k < N; k++)
        {
            int product = A[row*N + k] * B[k*N + col]; // Map
            sum += product;                            // Reduce
        }

        C[row*N + col] = sum;
    }
}

int main()
{
    int N;
    printf("Enter matrix size: ");
    scanf("%d",&N);

    int size = N*N*sizeof(int);

    int *A = (int*)malloc(size);
    int *B = (int*)malloc(size);
    int *C = (int*)malloc(size);

    // Initialize matrices
    for(int i=0;i<N*N;i++)
    {
        A[i] = 5;
        B[i] = 7;
    }

    // CPU computation
    clock_t c1 = clock();

    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++)
        {
            int sum = 0;
            for(int k=0;k<N;k++)
                sum += A[i*N+k] * B[k*N+j];

            C[i*N+j] = sum;
        }

    clock_t c2 = clock();
    double cpu_time = (double)(c2-c1)/CLOCKS_PER_SEC;

    // GPU
    int *dA,*dB,*dC;

    cudaMalloc(&dA,size);
    cudaMalloc(&dB,size);
    cudaMalloc(&dC,size);

    cudaMemcpy(dA,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,size,cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks((N+15)/16,(N+15)/16);

    cudaEvent_t s,e;
    cudaEventCreate(&s);
    cudaEventCreate(&e);

    cudaEventRecord(s);

    mapReduceMatMul<<<blocks,threads>>>(dA,dB,dC,N);

    cudaEventRecord(e);
    cudaEventSynchronize(e);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time,s,e);

    cudaMemcpy(C,dC,size,cudaMemcpyDeviceToHost);

    // Print only 4x4 output
    printf("\nTop-left 4x4 result:\n");
    for(int i=0;i<4;i++)
    {
        for(int j=0;j<4;j++)
            printf("%d ",C[i*N+j]);
        printf("\n");
    }

    printf("\nCPU Time = %f sec",cpu_time);
    printf("\nGPU Time = %f ms\n",gpu_time);

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);

    free(A);
    free(B);
    free(C);

    return 0;
}


Writing mapreduce_matmul.cu


In [2]:
!nvcc mapreduce_matmul.cu -o mapred
!./mapred

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter matrix size: 5

Top-left 4x4 result:
175 175 175 175 
175 175 175 175 
175 175 175 175 
175 175 175 175 

CPU Time = 0.000002 sec
GPU Time = 98.419617 ms
